# 03. 베이스라인 CNN — 불균형 대응 3종 비교

직접 만든 CNN 하나를 **동일한 시드·구조·에폭**으로 세 번 학습시켜 비교합니다.

| 실험 | 배치 구성 | 손실 함수 | 강의에서 배운 것 |
|---|---|---|---|
| `none` | `shuffle=True` | `CrossEntropyLoss()` | O (05 노트북 그대로) |
| `class_weight` | `shuffle=True` | `CrossEntropyLoss(weight=w)` | X |
| `sampler` | `WeightedRandomSampler` | `CrossEntropyLoss()` | X |

**비교의 핵심**: `none`은 accuracy가 가장 높게 나올 가능성이 큽니다.
전체의 34%인 clothes에 쏠리기 때문입니다. 그럼에도 `trash`, `brown-glass` 같은
소수 클래스의 recall이 낮다면 실용적으로는 실패한 모델입니다.
**"정확도가 떨어졌지만 더 좋은 모델"**을 숫자로 보여주는 것이 이 노트북의 목적입니다.

> **test split은 이 노트북에서 건드리지 않습니다.** 모델 선택은 val로만 하고,
> test는 `05_eval_report`에서 단 한 번 사용합니다. 여기서 test를 보면
> 그 시점부터 test가 더 이상 "본 적 없는 데이터"가 아니게 됩니다.

## 1. 환경 — GPU 강제

In [ ]:
import json
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch import amp
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (confusion_matrix, classification_report,
                             f1_score, recall_score)

# CPU로 조용히 넘어가는 것을 막는다 — 강의 코드의 else 'cpu' 패턴을 쓰지 않음
assert torch.cuda.is_available(), "CUDA 미탐지. 커널이 .venv인지 확인하세요."
device = torch.device("cuda")
torch.backends.cudnn.benchmark = True

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"

print("torch      :", torch.__version__)
print("GPU        :", torch.cuda.get_device_name(0))
print("capability :", torch.cuda.get_device_capability(0))
print("VRAM       : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

## 2. 설정값

In [ ]:
OUT = Path("outputs/garbage")
(OUT / "models").mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 128     # 베이스라인은 128 — 04의 전이학습(224)과 다름을 보고서에 명시할 것
BATCH_SIZE  = 64
EPOCHS      = 15
LR          = 1e-3
NUM_WORKERS = 0       # Windows 주피터에서 우선 0으로 시작. 아래 3-2 참고
SEED        = 42

def set_seed(seed=SEED):
    """세 실험의 출발선을 동일하게 맞춘다 — 비교의 전제"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

split = pd.read_csv(OUT / "metrics" / "split.csv")
label_map = json.load(open(OUT / "metrics" / "label_map.json", encoding="utf-8"))
stats = json.load(open(OUT / "metrics" / "norm_stats.json", encoding="utf-8"))
cw_df = pd.read_csv(OUT / "metrics" / "class_weights.csv").sort_values("label_idx")

classes = [c for c, _ in sorted(label_map.items(), key=lambda kv: kv[1])]
N_CLASSES = len(classes)

train_df = split[split.split == "train"].reset_index(drop=True)
val_df   = split[split.split == "val"].reset_index(drop=True)

print(f"train {len(train_df):,} | val {len(val_df):,} | classes {N_CLASSES}")
print("정규화(dataset):", stats["dataset_mean"], stats["dataset_std"])

## 3. 데이터셋

### 3-1. `.convert("RGB")`가 필수인 이유

EDA에서 **팔레트 모드(P) 이미지 34장**을 확인했습니다. 이걸 그대로 텐서로 만들면
채널 수가 3이 아니라 배치 안에서 shape이 어긋나 학습이 중단됩니다.

### 3-2. 증강 — 색은 건드리지 않는다

EDA의 Hue 겹침 계수가 세 쌍 모두 0.5 미만이었습니다. 즉 유리 3종은 **색이 판별 신호**입니다.
강의(05)의 CIFAR-10 설정은 `saturation=0.2`였지만, 여기서 채도를 그만큼 흔들면
white-glass(평균 채도 20)와 brown/green(60대)의 경계가 뭉개집니다.

- `saturation`은 0.05로 축소, `hue`는 0으로 고정
- `brightness`·`contrast`는 촬영 조건 차이를 흡수하는 용도이므로 유지
- **val에는 증강을 넣지 않습니다.**

In [ ]:
mean, std = stats["dataset_mean"], stats["dataset_std"]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.05, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


class GarbageDataset(Dataset):
    def __init__(self, df, transform):
        self.paths     = df["path"].tolist()
        self.targets   = df["label_idx"].tolist()   # sampler가 이 리스트를 사용
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")   # P모드 34장 대응
        return self.transform(img), self.targets[i]


train_ds = GarbageDataset(train_df, train_tf)
val_ds   = GarbageDataset(val_df,   eval_tf)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

x, y = train_ds[0]
print("샘플 텐서:", x.shape, x.dtype, "| 라벨:", y, classes[y])

> **`NUM_WORKERS` 튜닝**: Windows 주피터는 워커를 spawn으로 띄우기 때문에
> 노트북 셀에 정의한 `GarbageDataset`을 못 찾아 멈추는 경우가 있습니다.
> 먼저 `0`으로 1에폭 돌려 시간을 재고, `4`로 올려 비교하세요.
> 멈추면 `GarbageDataset`만 `dataset.py`로 빼서 import하면 해결됩니다.
> 학습 중 `nvidia-smi -l 2`에서 GPU 사용률이 30%대로 출렁이면 데이터 로딩이 병목입니다.

## 4. 모델

강의(`05_합성곱신경망`)의 `CIFAR10_CNN` 구조를 따릅니다.
Conv → BatchNorm → ReLU → MaxPool → Dropout2d 블록을 쌓는 형태입니다.

한 가지만 바꿨습니다. 강의는 `nn.Linear(2048, 10)`처럼 평탄화 크기를 직접 계산해
하드코딩했는데, 여기서는 `AdaptiveAvgPool2d(1)`을 씁니다. 입력 크기를 128에서 바꿔도
코드를 고칠 필요가 없고, 파라미터도 크게 줄어 과적합에 유리합니다.

입력 128 기준 공간 크기: `128 → 64 → 32 → 16 → 8` → AdaptiveAvgPool → `256`차원

In [ ]:
class GarbageCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()

        def block(cin, cout, drop=0.25):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2, 2),
                nn.Dropout2d(drop),
            )

        self.features = nn.Sequential(
            block(3,   32),
            block(32,  64),
            block(64,  128),
            block(128, 256),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)


def init_weights(model):
    """강의 05의 Kaiming 초기화"""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
    return model


_m = GarbageCNN()
print(_m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE)).shape, "<- (2, 12)이어야 정상")
print(f"파라미터 수: {sum(p.numel() for p in _m.parameters()):,}")
del _m

## 5. 학습·평가 루프

강의의 `train_epoch` / `evaluate_model` 구조를 유지하되 두 가지를 바꿨습니다.

1. **`if i >= 10: break` 제거** — 강의 코드의 그 줄은 수업 시간 단축용입니다.
   그대로 두면 배치 10개만 학습하고 끝납니다.
2. **AMP(혼합정밀도) 적용** — 8GB VRAM에서 메모리가 절반으로 줄고 학습이 빨라집니다.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with amp.autocast("cuda", dtype=torch.float16):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * labels.size(0)
        correct  += outputs.argmax(1).eq(labels).sum().item()
        total    += labels.size(0)

    return loss_sum / total, 100 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, total = 0.0, 0
    preds, trues = [], []

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with amp.autocast("cuda", dtype=torch.float16):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        loss_sum += loss.item() * labels.size(0)
        total    += labels.size(0)
        preds.append(outputs.argmax(1).cpu())
        trues.append(labels.cpu())

    y_pred = torch.cat(preds).numpy()
    y_true = torch.cat(trues).numpy()

    return {
        "loss":     loss_sum / total,
        "acc":      100 * (y_pred == y_true).mean(),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "y_pred":   y_pred,
        "y_true":   y_true,
    }

## 6. 실험 러너

세 실험의 차이는 **loader와 criterion 두 줄뿐**입니다.
모델 구조·초기화·옵티마이저·스케줄러·에폭·시드는 전부 동일합니다.
그래야 성능 차이를 불균형 대응 방식 탓으로 돌릴 수 있습니다.

**모델 선택 기준은 val accuracy가 아니라 val macro-F1입니다.**
accuracy로 고르면 clothes에 쏠린 체크포인트가 뽑힙니다.

In [ ]:
def build_loader_and_criterion(strategy):
    """strategy: 'none' | 'class_weight' | 'sampler'"""
    if strategy == "none":
        loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
        criterion = nn.CrossEntropyLoss()

    elif strategy == "class_weight":
        loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
        w = torch.tensor(cw_df["class_weight"].values,
                         dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=w)

    elif strategy == "sampler":
        per_class = cw_df.set_index("label_idx")["class_weight"].to_dict()
        sample_w  = [per_class[t] for t in train_ds.targets]
        sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w),
                                        replacement=True)
        # sampler와 shuffle은 동시에 줄 수 없음
        loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                            num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
        criterion = nn.CrossEntropyLoss()

    else:
        raise ValueError(strategy)

    return loader, criterion


def run_experiment(strategy, epochs=EPOCHS):
    set_seed(SEED)                       # 세 실험의 출발선을 동일하게
    model = init_weights(GarbageCNN()).to(device)

    train_loader, criterion = build_loader_and_criterion(strategy)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = amp.GradScaler("cuda")

    ckpt = OUT / "models" / f"baseline_{strategy}.pt"
    best_f1, history = -1.0, []
    t0 = time.time()

    print(f"\n{'=' * 62}\n[{strategy}]  epochs={epochs}  batches/epoch={len(train_loader)}\n{'=' * 62}")

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, scaler)
        va = evaluate(model, val_loader, criterion)
        scheduler.step()

        history.append({"epoch": ep, "train_loss": tr_loss, "train_acc": tr_acc,
                        "val_loss": va["loss"], "val_acc": va["acc"],
                        "val_macro_f1": va["macro_f1"],
                        "lr": optimizer.param_groups[0]["lr"]})

        star = ""
        if va["macro_f1"] > best_f1:      # 선택 기준 = macro-F1
            best_f1 = va["macro_f1"]
            torch.save({"model": model.state_dict(), "strategy": strategy,
                        "epoch": ep, "val_macro_f1": best_f1,
                        "classes": classes, "img_size": IMG_SIZE}, ckpt)
            star = "  *best"

        print(f"  ep{ep:02d}  train {tr_loss:.3f}/{tr_acc:5.1f}%  "
              f"val {va['loss']:.3f}/{va['acc']:5.1f}%  macroF1 {va['macro_f1']:.4f}{star}")

    elapsed = time.time() - t0

    # best 체크포인트를 다시 불러 최종 지표 산출
    model.load_state_dict(torch.load(ckpt)["model"])
    final = evaluate(model, val_loader, criterion)
    recalls = recall_score(final["y_true"], final["y_pred"],
                           average=None, labels=range(N_CLASSES), zero_division=0)

    print(f"  완료 {elapsed/60:.1f}분 | best macroF1 {best_f1:.4f}")

    return {"strategy": strategy, "history": pd.DataFrame(history),
            "final": final, "recalls": recalls,
            "minutes": elapsed / 60, "best_f1": best_f1}

## 7. 실행

세 실험이 순차 실행됩니다. 1에폭 소요시간을 먼저 확인하고 싶으시면
아래 셀을 `run_experiment("none", epochs=1)`로 한 번 돌려 시간을 재보세요.
15에폭 × 3실험이므로 1에폭이 1분이면 전체 45분입니다.

In [ ]:
# 시간 측정용 — 필요 없으면 건너뛰세요
_probe = run_experiment("none", epochs=1)
print(f"\n1에폭 = {_probe['minutes']:.1f}분  →  15에폭 × 3실험 ≈ {_probe['minutes'] * 45:.0f}분 예상")
del _probe

In [ ]:
results = {}
for strategy in ["none", "class_weight", "sampler"]:
    results[strategy] = run_experiment(strategy)
    torch.cuda.empty_cache()

## 8. 비교 — 여기가 보고서의 본문입니다

In [ ]:
rows = []
for s, r in results.items():
    f = r["final"]
    rows.append({
        "strategy":     s,
        "val_acc":      round(f["acc"], 2),
        "macro_f1":     round(f["macro_f1"], 4),
        "min_recall":   round(float(r["recalls"].min()), 3),
        "worst_class":  classes[int(r["recalls"].argmin())],
        "recall_std":   round(float(r["recalls"].std()), 3),
        "minutes":      round(r["minutes"], 1),
    })

summary = pd.DataFrame(rows)
summary.to_csv(OUT / "metrics" / "baseline_comparison.csv", index=False, encoding="utf-8")
print(summary.to_string(index=False))

**읽는 법**

- `val_acc`와 `macro_f1`이 **반대로 움직이면** 그게 이 과제의 핵심 발견입니다.
- `min_recall`: 가장 못 맞힌 클래스의 재현율. 실용성의 하한선입니다.
- `recall_std`: 클래스별 재현율의 표준편차. **낮을수록 고른 모델**입니다.
  불균형 대응의 목적이 바로 이 값을 낮추는 것입니다.

In [ ]:
rec = pd.DataFrame({s: results[s]["recalls"] for s in results}, index=classes)
rec = rec.round(3)

# 지지도(val 장수) 함께 표시
rec.insert(0, "val_n", val_df["label_idx"].value_counts().reindex(range(N_CLASSES)).values)
print("클래스별 recall (val)")
print(rec.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(N_CLASSES)
width = 0.26
colors = {"none": "#95a5a6", "class_weight": "#4a7ba7", "sampler": "#c0392b"}

for i, s in enumerate(results):
    ax.bar(x + (i - 1) * width, results[s]["recalls"], width, label=s, color=colors[s])

ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_ylabel("recall (val)")
ax.set_title("Per-class recall by imbalance strategy")
ax.axhline(0.5, ls="--", lw=1, color="gray")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures" / "07_recall_by_strategy.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, s in zip(axes, results):
    h = results[s]["history"]
    ax.plot(h["epoch"], h["train_acc"], label="train acc", color="#4a7ba7")
    ax.plot(h["epoch"], h["val_acc"],   label="val acc",   color="#c0392b")
    ax.plot(h["epoch"], h["val_macro_f1"] * 100, label="val macroF1×100",
            color="#27ae60", ls="--")
    ax.set_title(s); ax.set_xlabel("epoch"); ax.set_ylim(0, 100)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT / "figures" / "08_curves.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, s in zip(axes, results):
    f = results[s]["final"]
    cm = confusion_matrix(f["y_true"], f["y_pred"], labels=range(N_CLASSES))
    cmn = cm / cm.sum(axis=1, keepdims=True)      # 행 정규화 = 클래스별 recall
    sns.heatmap(cmn, ax=ax, cmap="Blues", vmin=0, vmax=1, cbar=(s == "sampler"),
                xticklabels=classes, yticklabels=classes, square=True,
                annot=False)
    ax.set_title(f"{s}  (macroF1 {f['macro_f1']:.3f})")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")

plt.tight_layout()
plt.savefig(OUT / "figures" / "09_confusion.png", dpi=150)
plt.show()

**혼동행렬에서 확인할 것**

행 정규화했으므로 **대각선 = 클래스별 recall**입니다.

EDA에서 유리 3종의 Hue 겹침 계수가 0.5 미만이었습니다. 즉 색으로 구분 가능하다는 뜻이므로
**유리 3×3 블록의 대각선이 진하게 나오는 것이 정상**입니다. 만약 여기가 흐리면
모델이 색 정보를 제대로 못 쓰고 있다는 신호이고, 증강이 과했거나 학습이 부족한 것입니다.

반면 `brown-glass ↔ white-glass`(겹침 0.422로 가장 높았음)에서 오분류가 나온다면
EDA의 예측이 맞아떨어진 것이므로 그대로 보고서에 쓰면 됩니다.

In [ ]:
best = summary.sort_values("macro_f1", ascending=False).iloc[0]
print("최종 선택:", best["strategy"], f"(macro_f1 {best['macro_f1']})")
print()
r = results[best["strategy"]]["final"]
print(classification_report(r["y_true"], r["y_pred"],
                            target_names=classes, digits=3, zero_division=0))

In [ ]:
# 04에서 이어붙일 실험 기록
exp_path = OUT / "metrics" / "experiments.csv"
log = summary.assign(model="GarbageCNN(scratch)", img_size=IMG_SIZE, epochs=EPOCHS)

if exp_path.exists():
    prev = pd.read_csv(exp_path)
    log = pd.concat([prev, log], ignore_index=True).drop_duplicates(
        subset=["model", "strategy"], keep="last")

log.to_csv(exp_path, index=False, encoding="utf-8")
print(log.to_string(index=False))

for s, r in results.items():
    r["history"].to_csv(OUT / "metrics" / f"history_baseline_{s}.csv",
                        index=False, encoding="utf-8")
print("\n체크포인트:", *[p.name for p in sorted((OUT / 'models').glob('baseline_*.pt'))])

---

## 다음 단계

여기서 나온 **가장 좋은 불균형 대응 방식 하나**를 골라 `04_transfer`에서 그대로 씁니다.
전이학습에서 세 방식을 또 세 번 돌릴 필요는 없습니다. 학습 시간이 훨씬 길고,
비교 자체는 이 노트북에서 이미 끝났기 때문입니다.

보고서에 넣을 문장 형태:

> 베이스라인 CNN에서 불균형 대응 3종을 비교한 결과 `___`이 macro-F1 기준 가장 우수했고,
> 특히 최소 클래스 recall이 `___` → `___`로 개선되었다. accuracy는 `___`p 하락했으나
> 이는 다수 클래스(clothes)에 대한 쏠림이 완화된 결과이다.

→ `03P_04_transfer.ipynb` 로 이동